# 05 — Pipeline Spark : Bronze → Silver → Gold (8 fichiers Suricata EVE JSON)

**Objectif** : reconstruire en PySpark, sur **les 8 fichiers complets (~193 Go)**, le jeu ML du prototype pandas.

| Couche | Contenu |
|---|---|
| **Bronze** | événements `alert`, `flow`, `http` typés et aplatis, **un dossier par fichier source**, partitionnés par date |
| **Silver** | une ligne par flux alerté : labels faibles v1/v2, signature la plus sévère, adresses du flux, direction, `only_noise` |
| **Gold** | variables ML (flux, temps, profil d'hôte sur le **passé strict**) + texte HTTP le plus proche de l'alerte |

**Choix de conception**
- Bronze est construit **fichier par fichier** (indépendant, reprise possible) ; Silver et Gold sont construits **une seule fois sur l'union des 8 fichiers** (sinon l'historique des IP repartirait à zéro à chaque fichier).
- Si la sortie d'une version précédente de ce notebook est attachée en *Input*, les fichiers Bronze déjà construits sont **réutilisés** (pas de ré-extraction).
- Le fichier **008** (tempête d'alertes de 51 min) est conservé mais **diagnostiqué** ; la colonne `source_file` permet de l'exclure à l'entraînement.

**Exécution** : *Accelerator* = None, *Internet* = On, Input = `dataset-suricata-logs` (+ optionnel : `AI CYBER` pour la validation, sortie de ce notebook pour réutiliser Bronze). Lancer avec **Save Version → Save & Run All** : durée estimée 1 h à 1 h 30 si Bronze est à reconstruire.

In [1]:
!pip install -q pyspark
!java -version 2>&1 | head -1

openjdk version "21.0.12.1" 2026-08-18


## 0. Configuration et session Spark

In [2]:
import os, glob, time, json, shutil
import pandas as pd
from pyspark.sql import SparkSession, Window, functions as F, types as T

T0 = time.time()
DATA_DIR = "/kaggle/input/datasets/muttaqien19/dataset-suricata-logs"
WORK = "/kaggle/working"
LAKE = f"{WORK}/lake"
INCLUDE_FILES = None          # None = les 8 fichiers ; ex. ["001"] pour un test rapide
NOISE = "Generic Protocol Command Decode"
SENSITIVE_RE = r"(?i)(Hidden Environment File|sftp-config|sftp\.json)"
os.makedirs("/tmp/spark", exist_ok=True)

spark = (SparkSession.builder
         .appName("suricata-lakehouse")
         .master("local[*]")
         .config("spark.driver.memory", "20g")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.files.maxPartitionBytes", str(256 * 1024 * 1024))
         .config("spark.sql.sources.partitionColumnTypeInference.enabled", "false")   # garde "001" en texte
         .config("spark.local.dir", "/tmp/spark")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

files = sorted(p for p in glob.glob(f"{DATA_DIR}/**/eve.json.*", recursive=True) if os.path.isfile(p))
assert files, "Aucun fichier eve.json trouvé : vérifier l'Input dataset-suricata-logs"
tags = [p.split(".")[-1] for p in files]
if INCLUDE_FILES:
    files = [p for p, t in zip(files, tags) if t in INCLUDE_FILES]
    tags = [p.split(".")[-1] for p in files]
print("Spark", spark.version, "| cœurs :", spark.sparkContext.defaultParallelism, "| fichiers :", tags)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/23 12:41:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/23 12:41:42 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


Spark 4.0.4 | cœurs : 4 | fichiers : ['001', '002', '003', '004', '005', '006', '007', '008']


## 1. Bronze : extraction fichier par fichier
Schéma explicite (pas d'inférence sur 24 Go), lignes JSON tronquées ignorées (`DROPMALFORMED`), une lecture du fichier pour les 3 tables (`cache`).

In [3]:
alert_t = T.StructType([T.StructField("signature_id", T.LongType()), T.StructField("signature", T.StringType()),
                        T.StructField("category", T.StringType()), T.StructField("severity", T.IntegerType()),
                        T.StructField("action", T.StringType())])
flow_t = T.StructType([T.StructField(c, T.LongType()) for c in
                       ["pkts_toserver", "pkts_toclient", "bytes_toserver", "bytes_toclient", "age"]] +
                      [T.StructField(c, T.StringType()) for c in ["start", "end", "state", "reason"]] +
                      [T.StructField("alerted", T.BooleanType())])
http_t = T.StructType([T.StructField(c, T.StringType()) for c in
                       ["hostname", "url", "http_user_agent", "http_method", "http_content_type", "protocol"]] +
                      [T.StructField("status", T.IntegerType()), T.StructField("length", T.LongType())])
schema = T.StructType([
    T.StructField("timestamp", T.StringType()), T.StructField("flow_id", T.LongType()),
    T.StructField("event_type", T.StringType()),
    T.StructField("src_ip", T.StringType()), T.StructField("src_port", T.IntegerType()),
    T.StructField("dest_ip", T.StringType()), T.StructField("dest_port", T.IntegerType()),
    T.StructField("proto", T.StringType()), T.StructField("app_proto", T.StringType()),
    T.StructField("alert", alert_t), T.StructField("flow", flow_t), T.StructField("http", http_t)])
base = ["ts", "date", "flow_id", "src_ip", "src_port", "dest_ip", "dest_port", "proto", "app_proto", "source_file"]
TABLES = ["alert", "flow", "http"]

# Bronze déjà construit dans une version précédente (Input) ? -> réutilisé en lecture seule
INPUT_BRONZE = sorted(glob.glob("/kaggle/input/**/lake/bronze", recursive=True))
def bronze_dir(name, tag):
    # Input (version terminée) : dossier présent avec des partitions date=... ; travail : _SUCCESS exigé
    for root in INPUT_BRONZE:
        d = f"{root}/{name}/file={tag}"
        if os.path.isdir(d) and any(x.startswith("date=") for x in os.listdir(d)):
            return d
    d = f"{LAKE}/bronze/{name}/file={tag}"
    return d if os.path.exists(f"{d}/_SUCCESS") else None

def build_bronze(path):
    tag = path.split(".")[-1]
    if all(bronze_dir(n, tag) for n in TABLES):
        print(f"{tag} : Bronze déjà présent ({bronze_dir('alert', tag).split('/lake/')[0]}), ignoré")
        return
    t0 = time.time()
    raw = (spark.read.schema(schema).option("mode", "DROPMALFORMED").json(path)
           .where(F.col("event_type").isin(*TABLES))
           .withColumn("ts", F.to_timestamp("timestamp", "yyyy-MM-dd'T'HH:mm:ss.SSSSSSZ"))
           .withColumn("date", F.to_date("ts"))
           .withColumn("source_file", F.lit(tag)))
    sub = {"alert": alert_t, "flow": flow_t, "http": http_t}
    raw.cache()
    for name in TABLES:
        (raw.where(F.col("event_type") == name)
            .select(*base, *[F.col(f"{name}.{c}").alias(c) for c in sub[name].names])
            .write.mode("overwrite").partitionBy("date").parquet(f"{LAKE}/bronze/{name}/file={tag}"))
    raw.unpersist()
    print(f"{tag} : Bronze construit en {(time.time() - t0) / 60:.1f} min")

print("Bronze en Input :", INPUT_BRONZE or "aucun")
for p in files:
    build_bronze(p)
print(f"Bronze prêt | {(time.time() - T0) / 60:.1f} min depuis le début")

Bronze en Input : aucun


001 : Bronze construit en 5.7 min


002 : Bronze construit en 5.2 min


003 : Bronze construit en 5.5 min


004 : Bronze construit en 6.4 min


005 : Bronze construit en 5.7 min


006 : Bronze construit en 5.9 min


007 : Bronze construit en 5.9 min


008 : Bronze construit en 7.5 min
Bronze prêt | 48.1 min depuis le début


In [7]:
from functools import reduce

def bronze(name):
    paths = [bronze_dir(name, t) for t in tags]
    assert all(paths), f"Bronze {name} incomplet : {dict(zip(tags, paths))}"
    # chaque fichier est lu séparément (ses partitions date=... sont reconnues), puis on réunit les 8
    parts = [spark.read.parquet(p).drop("date", "file") for p in paths]
    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), parts)
rows = []
for t in tags:
    r = {"fichier": t}
    for name in TABLES:
        d = spark.read.parquet(bronze_dir(name, t))
        a = d.agg(F.count("*").alias("n"), F.min("ts").alias("debut"), F.max("ts").alias("fin")).first()
        r[f"{name}_lignes"] = a["n"]
        if name == "alert":
            r["debut"], r["fin"] = str(a["debut"])[:16], str(a["fin"])[:16]
    rows.append(r)
bronze_stats = pd.DataFrame(rows)
print(bronze_stats.to_string(index=False))
print("\nTotaux :", bronze_stats[[f"{n}_lignes" for n in TABLES]].sum().to_dict())
!du -sh /kaggle/working/lake 2>/dev/null; df -h /kaggle/working | tail -1

fichier  alert_lignes            debut              fin  flow_lignes  http_lignes
    001       2791424 2024-03-12 07:56 2024-05-18 17:17     11432015      4483844
    002       2573943 2024-05-18 17:20 2024-07-05 06:09     10194482      5391393
    003       2204303 2024-07-05 06:09 2024-08-17 02:07     12186225      5232363
    004       4568539 2024-08-17 02:09 2024-09-20 18:20     13004518      4436073
    005       4911772 2024-09-20 18:20 2024-11-09 03:48      9910654      4176245
    006       2814380 2024-11-09 03:49 2024-12-29 11:12     13975085      3116278
    007       3740027 2024-12-30 02:36 2025-02-11 07:45     11015054      3565620
    008      27939935 2025-02-11 07:45 2025-02-11 08:36        10444         2894

Totaux : {'alert_lignes': 51544323, 'flow_lignes': 81728477, 'http_lignes': 30404710}
4.4G	/kaggle/working/lake
/dev/loop2       20G  4.4G   16G  23% /kaggle/working


## 2. Diagnostic du fichier 008 (tempête d'alertes)
27,9 M d'alertes en 51 minutes pour ~10 000 flux. On regarde ce qu'il contient avant de l'intégrer. Silver regroupe les alertes **par flux** : ce fichier ne produira au plus que quelques milliers de lignes en Gold, quel que soit son nombre d'alertes.

In [8]:
if "008" in tags:
    a8 = spark.read.parquet(bronze_dir("alert", "008"))
    n8 = a8.count()
    print("Alertes :", n8)
    print("Catégories :"); a8.groupBy("category").count().orderBy(F.desc("count")).show(6, truncate=60)
    print("Signatures :"); a8.groupBy("signature").count().orderBy(F.desc("count")).show(6, truncate=70)
    print("Couples source → destination :")
    a8.groupBy("src_ip", "dest_ip", "dest_port").count().orderBy(F.desc("count")).show(5)
    per_flow = a8.groupBy("flow_id").count()
    print("flow_id distincts :", per_flow.count(), "| alertes par flux (max) :", per_flow.agg(F.max("count")).first()[0])
    print("Doublons exacts :", n8 - a8.dropDuplicates().count())
else:
    print("Fichier 008 non inclus.")

Alertes : 27939935
Catégories :


+----------------------------------------------+--------+
|                                      category|   count|
+----------------------------------------------+--------+
|               Generic Protocol Command Decode|27939644|
|                                 Misc activity|     169|
|                        Web Application Attack|      52|
|                        Not Suspicious Traffic|      27|
|Device Retrieving External IP Address Detected|      20|
|                                   Misc Attack|       9|
+----------------------------------------------+--------+
only showing top 6 rows
Signatures :


+---------------------------------------------------------+--------+
|                                                signature|   count|
+---------------------------------------------------------+--------+
|                  SURICATA STREAM Packet with invalid ack|12822778|
|                  SURICATA STREAM ESTABLISHED invalid ack|12822755|
|         SURICATA STREAM ESTABLISHED packet out of window| 2290536|
|            SURICATA STREAM Packet with invalid timestamp|    2656|
|       SURICATA STREAM 3way handshake wrong seq wrong ack|     671|
|ET INFO User-Agent (python-requests) Inbound to Webserver|     159|
+---------------------------------------------------------+--------+
only showing top 6 rows
Couples source → destination :


+-------------+-------------+---------+--------+
|       src_ip|      dest_ip|dest_port|   count|
+-------------+-------------+---------+--------+
| 192.168.33.9|    10.10.7.4|    50439|12396176|
|    10.10.7.4| 192.168.33.9|     2222|10753595|
|103.187.88.26| 192.168.33.9|     8081| 2608710|
| 192.168.33.9|103.187.88.26|    52412|  732823|
| 192.168.33.9|103.187.88.26|    52914|  639622|
+-------------+-------------+---------+--------+
only showing top 5 rows


flow_id distincts : 494 | alertes par flux (max) : 23149771


Doublons exacts : 9668592


## 3. Silver : une ligne par flux alerté (union des 8 fichiers)
- Labels faibles au niveau de l'alerte : **v1** = sévérité ≤ 2 ou scan réseau ; **v2** = v1 ou fichier sensible (`.env`, `sftp-config`, `sftp.json`).
- Par flux : label maximal, **signature la plus sévère** via `min(struct(severity, ts, ...))` (une agrégation, robuste même si un flux a des millions d'alertes, contrairement à une fenêtre).
- **Adresses et ports du flux** (sens client → serveur), direction, `only_noise` = toutes les alertes du flux sont du bruit.

In [9]:
t0 = time.time()
al = (bronze("alert")
      .where(F.col("flow_id").isNotNull())
      .dropDuplicates()
      .withColumn("is_noise", (F.col("category") == NOISE).cast("int"))
      .withColumn("label_v1", ((F.col("severity") <= 2) |
                               (F.col("category") == "Detection of a Network Scan")).cast("int"))
      .withColumn("label_v2", ((F.col("label_v1") == 1) | F.col("signature").rlike(SENSITIVE_RE)).cast("int")))

agg = (al.groupBy("flow_id").agg(
          F.min("ts").alias("ts"), F.count("*").alias("n_alerts"),
          F.max("label_v1").alias("label_v1"), F.max("label_v2").alias("label_v2"),
          F.min("is_noise").alias("only_noise"),
          F.first("app_proto", ignorenulls=True).alias("app_proto"),
          F.min(F.struct("severity", "ts", "signature", "category")).alias("top"))
       .select("flow_id", "ts", "n_alerts", "label_v1", "label_v2", "only_noise", "app_proto",
               F.col("top.signature").alias("sig"), F.col("top.category").alias("sig_category"),
               F.col("top.severity").alias("sig_severity")))

flow_cols = ["flow_id", "src_ip", "dest_ip", "src_port", "dest_port", "proto", "pkts_toserver", "pkts_toclient",
             "bytes_toserver", "bytes_toclient", "age", "state", "reason", "source_file"]
fl1 = bronze("flow").select(*flow_cols).dropDuplicates(["flow_id"])
is_int = lambda c: F.col(c).startswith("192.168.")
silver = (agg.join(fl1, "flow_id", "inner")
             .withColumn("direction", F.when(~is_int("src_ip") & is_int("dest_ip"), "entrant")
                                      .when(is_int("src_ip") & ~is_int("dest_ip"), "sortant")
                                      .when(is_int("src_ip") & is_int("dest_ip"), "interne")
                                      .otherwise("externe"))
             .withColumn("date", F.to_date("ts")))
silver.write.mode("overwrite").partitionBy("direction").parquet(f"{LAKE}/silver/alert_flow")
print(f"Silver écrit en {(time.time() - t0) / 60:.1f} min")

s_all = spark.read.parquet(f"{LAKE}/silver/alert_flow")
n_agg = agg.count(); n_s = s_all.count()
print(f"Flux alertés : {n_agg:,} | avec enregistrement flow : {n_s:,} ({n_s / n_agg:.2%})")
print("Par direction, hors flux 100 % bruit :")
(s_all.where("only_noise = 0").groupBy("direction")
      .agg(F.count("*").alias("flux"), F.round(F.avg("label_v1"), 3).alias("taux_v1"),
           F.round(F.avg("label_v2"), 3).alias("taux_v2")).orderBy(F.desc("flux")).show())
print("Par fichier source (entrant, hors bruit) :")
(s_all.where("only_noise = 0 AND direction = 'entrant'").groupBy("source_file")
      .agg(F.count("*").alias("flux"), F.round(F.avg("label_v1"), 3).alias("taux_v1")).orderBy("source_file").show())
print("Flux 100 % bruit :", s_all.where("only_noise = 1").count())

Silver écrit en 10.0 min


Flux alertés : 3,639,186 | avec enregistrement flow : 3,639,114 (100.00%)
Par direction, hors flux 100 % bruit :


+---------+------+-------+-------+
|direction|  flux|taux_v1|taux_v2|
+---------+------+-------+-------+
|  entrant|981662|  0.279|  0.391|
|  interne|177041|  0.193|  0.783|
|  sortant| 46279|  0.973|  0.973|
+---------+------+-------+-------+

Par fichier source (entrant, hors bruit) :


+-----------+------+-------+
|source_file|  flux|taux_v1|
+-----------+------+-------+
|        001|103621|  0.259|
|        002|102480|  0.357|
|        003|131780|  0.373|
|        004|152359|  0.349|
|        005|186802|  0.184|
|        006|138789|  0.225|
|        007|165619|  0.256|
|        008|   212|  0.363|
+-----------+------+-------+

Flux 100 % bruit : 2434132


## 4. Gold (1/2) : variables et profil d'hôte
Profil d'hôte calculé sur **tous les flux** (~82 M) et **uniquement sur les heures strictement passées** : « as-of join » distribué (union requêtes + agrégats horaires, puis `last(ignorenulls)` sur une fenêtre par IP). Mêmes définitions que le prototype pandas.

In [10]:
t0 = time.time()
s = spark.read.parquet(f"{LAKE}/silver/alert_flow").where("only_noise = 0")
fl = bronze("flow")

hh = (fl.withColumn("t", F.date_trunc("hour", "ts"))
        .groupBy("src_ip", "t")
        .agg(F.count("*").alias("n_flows"),
             F.coalesce(F.sum(F.col("alerted").cast("int")), F.lit(0)).alias("n_alerted"),
             F.countDistinct("dest_port").alias("n_ports")))
wc = Window.partitionBy("src_ip").orderBy("t").rowsBetween(Window.unboundedPreceding, 0)
hh = hh.withColumn("cum_flows", F.sum("n_flows").over(wc)).withColumn("cum_alerted", F.sum("n_alerted").over(wc))

g = s.withColumn("t", F.date_trunc("hour", "ts"))
q = g.select("flow_id", "src_ip", "t", F.lit(1).alias("is_q"))
r = hh.select(F.lit(None).cast("long").alias("flow_id"), "src_ip", "t", F.lit(0).alias("is_q"),
              "cum_flows", "cum_alerted", "n_flows", "n_alerted", "n_ports", F.col("t").alias("t_src"))
u = q.unionByName(r, allowMissingColumns=True)
wl = (Window.partitionBy("src_ip").orderBy(F.col("t").asc(), F.col("is_q").desc())
            .rowsBetween(Window.unboundedPreceding, 0))       # à heure égale, l'alerte ne voit pas son heure
for c in ["cum_flows", "cum_alerted", "n_flows", "n_alerted", "n_ports", "t_src"]:
    u = u.withColumn(f"last_{c}", F.last(c, ignorenulls=True).over(wl))
prev = F.col("last_t_src") == (F.col("t") - F.expr("INTERVAL 1 HOUR"))
prof = u.where("is_q = 1").select(
    "flow_id",
    F.coalesce("last_cum_flows", F.lit(0)).alias("hist_flows"),
    F.coalesce("last_cum_alerted", F.lit(0)).alias("hist_alerted"),
    F.when(prev, F.col("last_n_flows")).otherwise(0).alias("prev1h_flows"),
    F.when(prev, F.col("last_n_alerted")).otherwise(0).alias("prev1h_alerted"),
    F.when(prev, F.col("last_n_ports")).otherwise(0).alias("prev1h_ports"))

wi = Window.partitionBy("src_ip").orderBy("ts", "flow_id")
gold = (g.join(prof, "flow_id")
         .withColumn("hour", F.hour("ts"))
         .withColumn("dow", (F.dayofweek("ts") + 5) % 7)                       # lundi = 0, comme pandas
         .withColumn("bytes_ratio", F.col("bytes_toserver") / (F.col("bytes_toclient") + 1))
         .withColumn("pkts_total", F.col("pkts_toserver") + F.col("pkts_toclient"))
         .withColumn("ip_prev_alerts", F.row_number().over(wi) - 1)
         .withColumn("ip_gap_s", F.coalesce(F.col("ts").cast("double") - F.lag(F.col("ts").cast("double")).over(wi),
                                            F.lit(-1.0)))
         .withColumn("hist_alert_rate", F.col("hist_alerted") / (F.col("hist_flows") + 1))
         .withColumn("is_new_host", (F.col("hist_flows") == 0).cast("int"))
         .withColumn("label", F.col("label_v1"))
         .drop("t"))
gold.write.mode("overwrite").partitionBy("direction").parquet(f"{LAKE}/gold/features")
print(f"Gold (variables) écrit en {(time.time() - t0) / 60:.1f} min")

gf = spark.read.parquet(f"{LAKE}/gold/features")
(gf.groupBy("direction").agg(F.count("*").alias("lignes"), F.round(F.avg("label"), 3).alias("taux_v1"),
                             F.round(F.avg("is_new_host"), 3).alias("part_nouvelles_ip"),
                             F.round(F.avg("hist_flows"), 0).alias("hist_flows_moyen"))
   .orderBy(F.desc("lignes")).show())

Gold (variables) écrit en 1.6 min


+---------+------+-------+-----------------+----------------+
|direction|lignes|taux_v1|part_nouvelles_ip|hist_flows_moyen|
+---------+------+-------+-----------------+----------------+
|  entrant|981662|  0.279|            0.089|          6747.0|
|  interne|177041|  0.193|              0.0|     1.4124066E7|
|  sortant| 46279|  0.973|              0.0|       3953533.0|
+---------+------+-------+-----------------+----------------+



## 5. Gold (2/2) : texte HTTP
Pour chaque alerte, la requête HTTP **du même flux** la plus proche dans le temps (±5 s) : `méthode URL || user-agent`. Le code de statut n'entre **pas** dans le texte (sinon le modèle recopierait la règle « 403 = suspect »).

In [11]:
t0 = time.time()
gf = spark.read.parquet(f"{LAKE}/gold/features")
h = bronze("http").select("flow_id", F.col("ts").alias("ts_http"), F.col("http_method").alias("method"),
                          "url", F.col("http_user_agent").alias("user_agent"), "status")
j = (gf.select("flow_id", "ts").join(h, "flow_id")
       .withColumn("dt_s", F.abs(F.col("ts").cast("double") - F.col("ts_http").cast("double")))
       .where("dt_s <= 5"))
wn = Window.partitionBy("flow_id").orderBy("dt_s", "ts_http")
txt = (j.withColumn("rk", F.row_number().over(wn)).where("rk = 1")
         .select("flow_id", "method", "url", "user_agent", "status", "dt_s",
                 F.trim(F.concat(F.coalesce("method", F.lit("")), F.lit(" "),
                                 F.substring(F.coalesce("url", F.lit("")), 1, 500), F.lit(" || "),
                                 F.substring(F.coalesce("user_agent", F.lit("")), 1, 300))).alias("text")))
gold_ml = gf.join(txt, "flow_id", "left").withColumn("has_http", F.col("text").isNotNull().cast("int"))
gold_ml.write.mode("overwrite").partitionBy("direction").parquet(f"{LAKE}/gold/ml_dataset")
print(f"Gold (ml_dataset) écrit en {(time.time() - t0) / 60:.1f} min")

Gold (ml_dataset) écrit en 0.7 min


## 6. Contrôles : volumes, dérive mensuelle, validation contre le prototype pandas

In [12]:
gm = spark.read.parquet(f"{LAKE}/gold/ml_dataset")
print("Par direction :")
(gm.groupBy("direction").agg(F.count("*").alias("lignes"), F.round(F.avg("label"), 3).alias("taux_v1"),
                             F.round(F.avg("has_http"), 3).alias("part_http")).orderBy(F.desc("lignes")).show())

print("Trafic entrant par mois (dérive) :")
monthly = (gm.where("direction = 'entrant'")
             .groupBy(F.date_format("ts", "yyyy-MM").alias("mois"))
             .agg(F.count("*").alias("alertes"), F.round(F.avg("label"), 3).alias("taux_v1"),
                  F.round(F.avg("has_http"), 3).alias("part_http"), F.countDistinct("src_ip").alias("ip"))
             .orderBy("mois").toPandas())
print(monthly.to_string(index=False))

print("\nTop familles positives (entrant) :")
(gm.where("direction = 'entrant' AND label = 1").groupBy("sig").count()
   .orderBy(F.desc("count")).show(10, truncate=75))

# Validation : le prototype pandas (fichier 001) doit être retrouvé à l'identique
pdp = glob.glob("/kaggle/input/**/ml_dataset.parquet", recursive=True)
if pdp:
    pdf = pd.read_parquet(pdp[0], columns=["flow_id", "label", "hist_flows", "bytes_ratio"])
    pdf["flow_id"] = pdf["flow_id"].astype("int64")
    ids = spark.createDataFrame(pdf[["flow_id"]])
    sp = (gm.join(F.broadcast(ids), "flow_id")
            .select("flow_id", F.col("label").alias("label_spark"), F.col("hist_flows").alias("hist_flows_spark"),
                    F.col("bytes_ratio").alias("bytes_ratio_spark")).toPandas())
    c = pdf.merge(sp, on="flow_id", how="left")
    found = c["label_spark"].notna()
    print(f"\nValidation pandas (001) | flux retrouvés : {found.mean():.1%} | positifs retrouvés : "
          f"{found[c['label'] == 1].mean():.1%} | accord labels : {(c.loc[found, 'label'] == c.loc[found, 'label_spark']).mean():.2%}"
          f" | bytes_ratio identique : {(abs(c.loc[found, 'bytes_ratio'] - c.loc[found, 'bytes_ratio_spark']) < 1e-6).mean():.2%}")
    print("Remarque : hist_flows diffère désormais du prototype pour les flux de mi-mai, car l'historique couvre les 8 fichiers.")
else:
    print("\n(Validation pandas ignorée : attacher AI CYBER en Input pour l'activer.)")

Par direction :
+---------+------+-------+---------+
|direction|lignes|taux_v1|part_http|
+---------+------+-------+---------+
|  entrant|981662|  0.279|    0.839|
|  interne|177041|  0.193|     0.78|
|  sortant| 46279|  0.973|    0.016|
+---------+------+-------+---------+

Trafic entrant par mois (dérive) :


   mois  alertes  taux_v1  part_http   ip
2024-03    44977    0.188      0.864 3050
2024-04    36741    0.319      0.699 4501
2024-05    39082    0.328      0.695 4764
2024-06    66510    0.405      0.836 4879
2024-07   101002    0.412      0.846 5127
2024-08    95009    0.202      0.844 5318
2024-09   153018    0.332      0.901 6798
2024-10   114905    0.192      0.826 6319
2024-11    75691    0.238      0.731 4907
2024-12    96157    0.216      0.837 4720
2025-01    92848    0.265      0.882 4846
2025-02    65722    0.257      0.921 2715

Top familles positives (entrant) :
+---------------------------------------------------------------------------+-----+
|                                                                        sig|count|
+---------------------------------------------------------------------------+-----+
|                                               GPL WEB_SERVER 403 Forbidden|66299|
|                                        ET WEB_SERVER PHP tags in HTTP POST|30164


Validation pandas (001) | flux retrouvés : 84.1% | positifs retrouvés : 100.0% | accord labels : 100.00% | bytes_ratio identique : 100.00%
Remarque : hist_flows diffère désormais du prototype pour les flux de mi-mai, car l'historique couvre les 8 fichiers.


## 7. Export compact pour le ML et le dashboard
Un seul fichier Parquet, **trafic entrant uniquement** (périmètre de référence, sans les doublons du proxy), avec les colonnes utiles. C'est le fichier à versionner avec **DVC**.

In [13]:
keep = ["flow_id", "ts", "date", "source_file", "src_ip", "dest_ip", "src_port", "dest_port", "proto", "app_proto",
        "state", "reason", "pkts_toserver", "pkts_toclient", "bytes_toserver", "bytes_toclient", "age",
        "bytes_ratio", "pkts_total", "hour", "dow", "ip_prev_alerts", "ip_gap_s",
        "hist_flows", "hist_alerted", "hist_alert_rate", "is_new_host", "prev1h_flows", "prev1h_alerted", "prev1h_ports",
        "n_alerts", "sig", "sig_category", "sig_severity", "label_v1", "label_v2", "label",
        "has_http", "method", "url", "user_agent", "status", "text"]
out_dir = f"{WORK}/gold_export_tmp"
(spark.read.parquet(f"{LAKE}/gold/ml_dataset").where("direction = 'entrant'")
      .select(*keep).orderBy("ts").coalesce(1).write.mode("overwrite").parquet(out_dir))
part = glob.glob(f"{out_dir}/part-*.parquet")[0]
shutil.move(part, f"{WORK}/gold_ml_entrant.parquet")
shutil.rmtree(out_dir)

chk = pd.read_parquet(f"{WORK}/gold_ml_entrant.parquet", columns=["ts", "label", "has_http", "source_file"])
summary = {
    "fichiers": tags,
    "bronze": bronze_stats.to_dict(orient="records"),
    "gold_entrant_lignes": int(len(chk)),
    "periode": [str(chk["ts"].min()), str(chk["ts"].max())],
    "taux_v1": round(float(chk["label"].mean()), 4),
    "part_http": round(float(chk["has_http"].mean()), 4),
    "lignes_par_fichier": chk["source_file"].value_counts().sort_index().to_dict(),
    "duree_totale_min": round((time.time() - T0) / 60, 1),
}
json.dump(summary, open(f"{WORK}/lakehouse_summary.json", "w"), indent=1, ensure_ascii=False, default=str)
print(json.dumps({k: v for k, v in summary.items() if k != "bronze"}, indent=1, ensure_ascii=False, default=str))
print(f"\ngold_ml_entrant.parquet : {os.path.getsize(f'{WORK}/gold_ml_entrant.parquet') / 1e6:.0f} Mo")
!du -sh /kaggle/working/lake/* /kaggle/working/*.parquet 2>/dev/null

{
 "fichiers": [
  "001",
  "002",
  "003",
  "004",
  "005",
  "006",
  "007",
  "008"
 ],
 "gold_entrant_lignes": 981662,
 "periode": [
  "2024-03-12 14:21:45.483840",
  "2025-02-11 08:34:02.426896"
 ],
 "taux_v1": 0.279,
 "part_http": 0.8393,
 "lignes_par_fichier": {
  "001": 103621,
  "002": 102480,
  "003": 131780,
  "004": 152359,
  "005": 186802,
  "006": 138789,
  "007": 165619,
  "008": 212
 },
 "duree_totale_min": 81.9
}

gold_ml_entrant.parquet : 69 Mo
4.4G	/kaggle/working/lake/bronze
164M	/kaggle/working/lake/gold
152M	/kaggle/working/lake/silver
66M	/kaggle/working/gold_ml_entrant.parquet
